## **Importación de Librerías**

In [11]:
import os
import json
import pandas as pd
from google.cloud import storage, bigquery
from datetime import datetime


## 🔹Crear el Pipeline en Vertex AI
Creamos un pipeline en Vertex AI Pipelines para ejecutar el notebook de transformación.

In [12]:
# Definimos las variables del proyecto
PROJECT_ID = "proyectofinalgogleyelp"
BUCKET_NAME = "dataset-pf-gyelp"
NOTEBOOK_PATH = "gs://proyectofinalgogleyelp-us-notebooks/ETL-Yelp.ipynb"  # Ruta del notebook de ETL en GCS
BQ_DATASET = "proyectofinalgogleyelp.proyecto_dw"
BQ_TABLE = "proyectofinalgogleyelp.proyecto_dw.dim_business"

# Inicializamos clientes de GCP
storage_client = storage.Client(PROJECT_ID)
bq_client = bigquery.Client(PROJECT_ID)

# Función para verificar si hay nuevos datos en GCS
def check_new_data(bucket_name):
    """Verifica si hay nuevos archivos de datos en el bucket."""
    bucket = storage_client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix="data/")
    new_files = [blob.name for blob in blobs if "processed" not in blob.name]
    
    return new_files

# Función para cargar datos en un DataFrame
def load_data_from_gcs(file_path):
    """Carga datos desde un archivo en GCS a un DataFrame de pandas."""
    file_uri = f"gs://{BUCKET_NAME}/{file_path}"
    df = pd.read_parquet(file_uri)
    return df

# Función para limpiar y transformar los datos
def transform_data(df):
    """Aplica transformaciones a los datos."""
    df["load_timestamp"] = datetime.utcnow()  # Agregamos la fecha de carga
    df.drop_duplicates(inplace=True)  # Eliminamos duplicados
    
    return df

# Función para realizar la carga incremental en BigQuery
def load_to_bigquery(df, dataset, table):
    """Carga los datos en BigQuery con la opción de carga incremental."""
    table_id = f"{PROJECT_ID}.{dataset}.{table}"
    
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,  # Agregamos datos sin sobrescribir
        source_format=bigquery.SourceFormat.PARQUET
    )
    
    job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()  # Esperamos a que termine la carga
    
    print(f"Datos cargados en {table_id}")

# Función principal de la pipeline
def etl_pipeline():
    """Ejecuta el proceso ETL de carga incremental."""
    new_files = check_new_data(BUCKET_NAME)
    
    if not new_files:
        print("No hay nuevos datos para procesar.")
        return
    
    for file_path in new_files:
        print(f"Procesando archivo: {file_path}")
        
        df = load_data_from_gcs(file_path)
        df = transform_data(df)
        load_to_bigquery(df, BQ_DATASET, BQ_TABLE)
        
        # Marcamos el archivo como procesado (opcionalmente renombrándolo)
        bucket = storage_client.bucket(BUCKET_NAME)
        blob = bucket.blob(file_path)
        new_name = file_path.replace("data/", "processed_data/")
        bucket.rename_blob(blob, new_name)
        print(f"Archivo {file_path} movido a {new_name}")

# Ejecutamos la pipeline
etl_pipeline()


No hay nuevos datos para procesar.
